# TC_09: FCNN with seed variation
Train FCNN with 5 different random seeds on clean. Run full diagnostic pipeline at each level and compare against TC_03 (Golden Baseline)

In [1]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import torch
from sklearn.model_selection import train_test_split

sys.path.append('..')
os.chdir('..')
plt.style.use('default')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

from src.utils.config_loader import load_config
from src.utils.data_loader import load_tabular_data
from src.data_diagnostics.quality_checks import check_tabular_quality
from src.utils.visualization import plot_class_distribution, plot_correlation, plot_feature_boxplots
from src.utils.performance_tracker import PerformanceTracker
from src.models.tabular_models import train_fcnn, evaluate_model_fcnn
from src.inference_diagnostics.uncertainty import deep_ensemble_prediction_tab, mc_dropout_prediction_tab
from src.inference_diagnostics.explainability import shap_tab, lime_tab,calculate_consistency_tabular
from src.inference_diagnostics.flagging import assign_flags,evaluate_flags
from src.utils.visualization import plot_uncertainty_distribution, plot_flag_distribution
from src.utils.report_generator import generate_report,save_report


config = load_config()


### Load clean data


In [2]:
X_train, X_test, y_train, y_test = load_tabular_data(config)
quality_report = check_tabular_quality(X_train, y_train, config)

# Load seed
seeds = config['random_seeds']['seed_variation_list']
num_seeds = config['stage2_training_stability']['seed_variation']['num_seeds']
seeds = seeds[:num_seeds]

Loading dataset.
No missing value found.
Data loaded for 202944 train, 50736 test
 Features: 21
EDA started for tabular data.
Samples: 202944, Features: 21
Class distribution: {0: 174667, 1: 28277}
Imbalance ratio: 0.1619
Duplicate rows: 18580
Total outlier count: 89295
EDA completed for Diabetes Health Indicators


### Run full pipeline for each data size fractions

In [3]:
all_results = {}
# Get golden baseling MC Dropout and Deep Ensemble threshold values
mc_threshold = config['stage3_inference']['flagging']['mc_dropout_uncertainty_threshold']
de_threshold = config['stage3_inference']['flagging']['deep_ensemble_uncertainty_threshold']

for seed in seeds:
    print(f"Seed variation: {seed}")

    tracker = PerformanceTracker()

    # Set seed
    torch.manual_seed(seed)
    np.random.seed(seed)

    # Train FCNN with seed variation
    tracker.start_performance_track()
    fcnn_model = train_fcnn(X_train, y_train, config)
    tracker.stop_performance_track(f"FCNN training seed variation: {seed} ")

    tracker.start_performance_track()
    fcnn_accuracy, fcnn_prediction, fcnn_report = evaluate_model_fcnn(fcnn_model, X_test, y_test, config)
    tracker.stop_performance_track(f"FCNN evaluation seed variation: {seed} ")

    # MC Dropout
    tracker.start_performance_track()
    mc_means_probabilities, mc_uncertainty = mc_dropout_prediction_tab(fcnn_model, X_test, config)
    tracker.stop_performance_track(f"FCNN MC Dropout seed variation: {seed} ")

    print(f"MC Dropout uncertainty status:")
    print(f"Mean: {mc_uncertainty.mean()}")
    print(f"Max: {mc_uncertainty.max()}")
    print(f" 90th percentile (threshold): {mc_threshold}")

    plot_uncertainty_distribution(mc_uncertainty, f"FCNN MC Dropout seed variation: {seed} ", mc_threshold, config)

    # Deep Ensemble
    tracker.start_performance_track()
    de_means_probabilities, de_uncertainty = deep_ensemble_prediction_tab(train_fcnn, X_train, y_train, X_test, config)
    tracker.stop_performance_track(f"FCNN Deep Ensemble seed variation: {seed} ")

    print(f"Deep Ensembl uncertainty status:")
    print(f"Mean: {de_uncertainty.mean()}")
    print(f"Max: {de_uncertainty.max()}")
    print(f" 90th percentile (threshold): {de_threshold}")

    plot_uncertainty_distribution(de_uncertainty, f"FCNN Deep Ensemble seed variation: {seed} ", de_threshold, config)

    # SHAP
    tracker.start_performance_track()
    shap_values, shap_samples = shap_tab(fcnn_model, X_train, X_test, config, is_pytorch = True)
    tracker.stop_performance_track(f"FCNN SHAP seed variation: {seed} ")
    print(f"SHAP values shape: {shap_values.shape}")

    # LIME
    tracker.start_performance_track()
    lime_explanations, lime_samples = lime_tab(fcnn_model, X_train, X_test, config, is_pytorch = True)
    tracker.stop_performance_track(f"FCNN LIME seed variation: {seed} ")
    print(f"LIME explanations: {len(lime_explanations)}")

    # Calculate consistency
    feature_names = list(X_train.columns)
    consistency_scores = calculate_consistency_tabular(shap_values, lime_explanations, feature_names, top=5)

    # MC Dropout with XAI flags
    mc_y_predictions = mc_means_probabilities[:len(consistency_scores)].argmax(axis = 1)
    mc_flags = assign_flags(mc_uncertainty[:len(consistency_scores)], consistency_scores, mc_threshold, 0.5)
    mc_flag_results = evaluate_flags(mc_flags, mc_y_predictions, y_test[:len(consistency_scores)])
    plot_flag_distribution(mc_flags, f"FCNN MC Dropout + XAI Flagging seed variation: {seed} ", config)

    # Deep Ensemble flags
    de_y_predictions = de_means_probabilities[:len(consistency_scores)].argmax(axis = 1)
    de_flags = assign_flags(de_uncertainty[:len(consistency_scores)], consistency_scores, de_threshold, 0.5)
    de_flag_results = evaluate_flags(de_flags, de_y_predictions, y_test[:len(consistency_scores)])
    plot_flag_distribution(de_flags, f"FCNN Deep Ensembles + XAI Flagging seed variation: {seed} ", config)

    # MC Dropout without XAI (constant consistency)
    con_consistency = np.ones(len(consistency_scores))
    mc_flags_uq_only = assign_flags(mc_uncertainty[:len(consistency_scores)], con_consistency, mc_threshold, 0.5)

    print(f"MC Dropout without XAI Flagging:")
    mc_only_flag_results = evaluate_flags(mc_flags_uq_only, mc_y_predictions, y_test[:len(consistency_scores)])
    plot_flag_distribution(mc_flags_uq_only, f"FCNN MC Dropout only seed variation: {seed} ", config)

    print(f"MC Dropout with XAI green accuracy: {mc_flag_results['GREEN']['accuracy']}")
    print(f"MC Dropout without XAI green accuracy: {mc_only_flag_results['GREEN']['accuracy']}")
    print(f"Improvement with XAI: {mc_flag_results['GREEN']['accuracy'] - mc_only_flag_results['GREEN']['accuracy']}")

    # Deep Ensemble without XAI (constant consistency)
    de_flags_uq_only = assign_flags(de_uncertainty[:len(consistency_scores)], con_consistency, de_threshold, 0.5)

    print(f"Deep Ensemble without XAI Flagging:")
    de_only_flag_results = evaluate_flags(de_flags_uq_only, de_y_predictions, y_test[:len(consistency_scores)])
    plot_flag_distribution(de_flags_uq_only, f"FCNN Deep Ensembles only seed variation: {seed} ", config)

    print(f"Deep Ensembles with XAI green accuracy: {de_flag_results['GREEN']['accuracy']}")
    print(f"Deep Ensembles without XAI green accuracy: {de_only_flag_results['GREEN']['accuracy']}")
    print(f"Improvement with XAI: {de_flag_results['GREEN']['accuracy'] - de_only_flag_results['GREEN']['accuracy']}")

    level_result = {
    'perturbation': 'seed variation',
    'seed': seed,
    'model': 'FCNN',
    'accuracy': round(fcnn_accuracy, 4),
    'classification_report': fcnn_report,
    'quality_report': quality_report['feature_stats'],
    'mc_uncertainty': {
        'mean': round(float(mc_uncertainty.mean()), 8),
        'max': round(float(mc_uncertainty.max()), 8),
        'threshold': mc_threshold
    },
    'de_uncertainty': {
        'mean': round(float(de_uncertainty.mean()), 8),
        'max': round(float(de_uncertainty.max()), 8),
        'threshold': de_threshold
    },
    'consistency':{
        'mean': round(float(consistency_scores.mean()), 4),
        'min': round(float(consistency_scores.min()), 4),
        'max': round(float(consistency_scores.max()), 4)
    },
    'flagging_mc': mc_flag_results,
    'flagging_de': de_flag_results,
    'flagging_mc_only': mc_only_flag_results,
    'mc_vs_mc_plus_xai': round(mc_flag_results['GREEN']['accuracy'] - mc_only_flag_results['GREEN']['accuracy'], 4),
    'flagging_de_only': de_only_flag_results,
    'de_vs_de_plus_xai': round(de_flag_results['GREEN']['accuracy'] - de_only_flag_results['GREEN']['accuracy'], 4),
    'performance': tracker.get_results()
    }

    all_results[f"{seed}"] = level_result

    # Save individual report
    report_output = generate_report(config,f'Diabetes - FCNN seed variation: {seed}', stage2_result = level_result )
    save_report(report_output, f'TC_09_FCNN_Seed_Variation_{seed}.json', config)


Seed variation: 0
FCNN training started.
Epoch 10/50
Epoch 20/50
Epoch 30/50
 Early stopping at epoch 39
FCNN training completed.
FCNN training seed variation: 0 : Time:111.10s, Memory:602.07MB
Model evaluation started for FCNN.
{'0': {'precision': 0.8789836525759577, 'recall': 0.9752215631941741, 'f1-score': 0.9246051131737502, 'support': 43667.0}, '1': {'precision': 0.5270979020979021, 'recall': 0.17060404583392277, 'f1-score': 0.2577749278614941, 'support': 7069.0}, 'accuracy': 0.8631149479659413, 'macro avg': {'precision': 0.7030407773369298, 'recall': 0.5729128045140485, 'f1-score': 0.5911900205176222, 'support': 50736.0}, 'weighted avg': {'precision': 0.8299557361038398, 'recall': 0.8631149479659413, 'f1-score': 0.8316962796044436, 'support': 50736.0}}
FCNN evaluation seed variation: 0 : Time:0.02s, Memory:13.39MB
MC Dropout started.
Pass 10/50 done.
Pass 20/50 done.
Pass 30/50 done.
Pass 40/50 done.
Pass 50/50 done.
MC Dropout finished for tabular.
FCNN MC Dropout seed variation

  0%|          | 0/200 [00:00<?, ?it/s]

SHAP finished. Explained: 200 samples.
FCNN SHAP seed variation: 0 : Time:7.28s, Memory:17.03MB
SHAP values shape: (200, 21, 2)
LIME started.
Explained 50 / 200 samples.
Explained 100 / 200 samples.
Explained 150 / 200 samples.
Explained 200 / 200 samples.
LIME finished. Explained: 200 samples.
FCNN LIME seed variation: 0 : Time:2.44s, Memory:13.25MB
LIME explanations: 200
Calculate consistency tabular.
RED: Count: 3 Accuracy:100.0%
YELLOW: Count: 42 Accuracy:71.4%
GREEN: Count: 155 Accuracy:90.3%
Flagging system is not working, needs threshold adjustment
Saved: results/figures\flagging_distribution_fcnn_mc_dropout_+_xai_flagging_seed_variation:_0_.png
RED: Count: 4 Accuracy:100.0%
YELLOW: Count: 44 Accuracy:68.2%
GREEN: Count: 152 Accuracy:92.1%
Flagging system is not working, needs threshold adjustment
Saved: results/figures\flagging_distribution_fcnn_deep_ensembles_+_xai_flagging_seed_variation:_0_.png
MC Dropout without XAI Flagging:
RED: Count: 0 Accuracy:0.0%
YELLOW: Count: 17 Ac

  0%|          | 0/200 [00:00<?, ?it/s]

SHAP finished. Explained: 200 samples.
FCNN SHAP seed variation: 1 : Time:8.01s, Memory:0.00MB
SHAP values shape: (200, 21, 2)
LIME started.
Explained 50 / 200 samples.
Explained 100 / 200 samples.
Explained 150 / 200 samples.
Explained 200 / 200 samples.
LIME finished. Explained: 200 samples.
FCNN LIME seed variation: 1 : Time:1.97s, Memory:14.18MB
LIME explanations: 200
Calculate consistency tabular.
RED: Count: 5 Accuracy:80.0%
YELLOW: Count: 42 Accuracy:81.0%
GREEN: Count: 153 Accuracy:88.9%
Flagging system is working
Saved: results/figures\flagging_distribution_fcnn_mc_dropout_+_xai_flagging_seed_variation:_1_.png
RED: Count: 5 Accuracy:80.0%
YELLOW: Count: 46 Accuracy:76.1%
GREEN: Count: 149 Accuracy:90.6%
Flagging system is working
Saved: results/figures\flagging_distribution_fcnn_deep_ensembles_+_xai_flagging_seed_variation:_1_.png
MC Dropout without XAI Flagging:
RED: Count: 0 Accuracy:0.0%
YELLOW: Count: 17 Accuracy:64.7%
GREEN: Count: 183 Accuracy:89.1%
Flagging system is wo

...

SHAP finished. Explained: 200 samples.
FCNN SHAP seed variation: 2 : Time:7.52s, Memory:0.00MB
SHAP values shape: (200, 21, 2)
LIME started.
Explained 50 / 200 samples.
Explained 100 / 200 samples.
Explained 150 / 200 samples.
Explained 200 / 200 samples.
LIME finished. Explained: 200 samples.
FCNN LIME seed variation: 2 : Time:2.50s, Memory:1.55MB
LIME explanations: 200
Calculate consistency tabular.
RED: Count: 7 Accuracy:57.1%
YELLOW: Count: 62 Accuracy:85.5%
GREEN: Count: 131 Accuracy:88.5%
Flagging system is working
Saved: results/figures\flagging_distribution_fcnn_mc_dropout_+_xai_flagging_seed_variation:_2_.png
RED: Count: 3 Accuracy:66.7%
YELLOW: Count: 70 Accuracy:81.4%
GREEN: Count: 127 Accuracy:90.6%
Flagging system is working
Saved: results/figures\flagging_distribution_fcnn_deep_ensembles_+_xai_flagging_seed_variation:_2_.png
MC Dropout without XAI Flagging:
RED: Count: 0 Accuracy:0.0%
YELLOW: Count: 21 Accuracy:57.1%
GREEN: Count: 179 Accuracy:89.9%
Flagging system is wor

  0%|          | 0/200 [00:00<?, ?it/s]

SHAP finished. Explained: 200 samples.
FCNN SHAP seed variation: 3 : Time:7.47s, Memory:0.00MB
SHAP values shape: (200, 21, 2)
LIME started.
Explained 50 / 200 samples.
Explained 100 / 200 samples.
Explained 150 / 200 samples.
Explained 200 / 200 samples.
LIME finished. Explained: 200 samples.
FCNN LIME seed variation: 3 : Time:1.84s, Memory:1.24MB
LIME explanations: 200
Calculate consistency tabular.
RED: Count: 3 Accuracy:66.7%
YELLOW: Count: 45 Accuracy:82.2%
GREEN: Count: 152 Accuracy:87.5%
Flagging system is working
Saved: results/figures\flagging_distribution_fcnn_mc_dropout_+_xai_flagging_seed_variation:_3_.png
RED: Count: 3 Accuracy:100.0%
YELLOW: Count: 53 Accuracy:75.5%
GREEN: Count: 144 Accuracy:91.0%
Flagging system is not working, needs threshold adjustment
Saved: results/figures\flagging_distribution_fcnn_deep_ensembles_+_xai_flagging_seed_variation:_3_.png
MC Dropout without XAI Flagging:
RED: Count: 0 Accuracy:0.0%
YELLOW: Count: 13 Accuracy:53.8%
GREEN: Count: 187 Accu

  0%|          | 0/200 [00:00<?, ?it/s]

SHAP finished. Explained: 200 samples.
FCNN SHAP seed variation: 4 : Time:7.96s, Memory:0.07MB
SHAP values shape: (200, 21, 2)
LIME started.
Explained 50 / 200 samples.
Explained 100 / 200 samples.
Explained 150 / 200 samples.
Explained 200 / 200 samples.
LIME finished. Explained: 200 samples.
FCNN LIME seed variation: 4 : Time:2.10s, Memory:0.00MB
LIME explanations: 200
Calculate consistency tabular.
RED: Count: 3 Accuracy:100.0%
YELLOW: Count: 54 Accuracy:83.3%
GREEN: Count: 143 Accuracy:87.4%
Flagging system is not working, needs threshold adjustment
Saved: results/figures\flagging_distribution_fcnn_mc_dropout_+_xai_flagging_seed_variation:_4_.png
RED: Count: 8 Accuracy:87.5%
YELLOW: Count: 56 Accuracy:76.8%
GREEN: Count: 136 Accuracy:91.2%
Flagging system is working
Saved: results/figures\flagging_distribution_fcnn_deep_ensembles_+_xai_flagging_seed_variation:_4_.png
MC Dropout without XAI Flagging:
RED: Count: 0 Accuracy:0.0%
YELLOW: Count: 9 Accuracy:55.6%
GREEN: Count: 191 Accur